# Этап 1: поиск по базе и ориентация GaAs

Нажмите **Run All**, затем подставьте свой файл и запустите ещё раз.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from ramancore.io import read_spectrum
from ramancore.library import load_library
from ramancore.matching import search
from ramancore.peaks import curves_on_grid
from ramancore.preprocess import process
from ramancore.semiconductors import gaas_orientation

lib = load_library("../data/library")   # ~1 с, база из ~9600 эталонов
print("Эталонов в базе:", len(lib["metadata"]))

In [ ]:
my_file = "../data/examples/synthetic_si.txt"  # ← подставьте свой файл
x, y, meta = read_spectrum(my_file, laser_nm=532)

## 1. Поиск по базе

`classes=["semiconductor"]` — искать только среди полупроводников; уберите, чтобы искать по всей базе.

In [ ]:
result = search(x, y, lib, top=10)
result.round(1)

In [ ]:
result_semi = search(x, y, lib, top=10, classes=["semiconductor"])
result_semi.round(1)

In [ ]:
# Наложение лучшего эталона на ваш спектр (оба после одинаковой обработки)
best = result.loc[0, "id"]
i = int(np.flatnonzero(lib["metadata"]["id"] == best)[0])
grid, u, _ = process(x, y, **lib["params"])
fig = go.Figure()
fig.add_trace(go.Scatter(x=grid, y=u, name="ваш спектр"))
fig.add_trace(go.Scatter(x=grid, y=lib["spectra"][i], name=f"{result.loc[0, 'material']} ({best})"))
fig.update_layout(xaxis_title="Рамановский сдвиг, см⁻¹", yaxis_title="Нормированная интенсивность")
fig.show()

## 2. Ориентация GaAs по I(TO)/I(LO)

Имеет смысл только для спектра GaAs.

In [ ]:
r = gaas_orientation(x, y)
print("Ориентация:", r["orientation"])
print(f"I(TO)/I(LO) = {r['ratio']:.3f} ± {r['ratio_err']:.3f}")
for w in r["warnings"]:
    print("⚠", w)
r["table"].round(2) if r["table"] is not None else None

In [ ]:
if r["fit"] is not None:
    fit = r["fit"]
    xf = np.linspace(fit["x"][0], fit["x"][-1], 1000)
    baseline, comps, total = curves_on_grid(r["table"], fit, xf)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fit["x"], y=fit["y"], mode="markers", name="данные"))
    for comp, name in zip(comps, ["TO", "LO"]):
        fig.add_trace(go.Scatter(x=xf, y=comp + baseline, name=name, line=dict(dash="dot")))
    fig.add_trace(go.Scatter(x=xf, y=total, name="модель"))
    fig.update_layout(xaxis_title="Рамановский сдвиг, см⁻¹", yaxis_title="Интенсивность")
    fig.show()